# Blood Rheology: Mt-ETV Model Analysis and Visualization (Logarithmic Coupling)

## Overview
The **thermodynamically consistent modified t-ETV (Mt-ETV) model** is a rheological model that describes blood behavior using three structural variables, derived from first principles via a Helmholtz free-energy formulation rather than empirical postulation:

1. **Cellular conformation tensor** ($\mathbf{c}^C$): Describes the deformation and orientation of individual red blood cells
2. **Rouleaux conformation tensor** ($\mathbf{c}^R$): Describes the deformation and orientation of cell aggregates (rouleaux formations)
3. **Structure parameter** ($\lambda$): Quantifies the degree of aggregation, ranging from 0 (fully dispersed) to 1 (fully aggregated)

Unlike the original t-ETV, the coupling between $\lambda$ and the rouleaux tensor is not postulated directly in the evolution equation. Instead, it emerges from a single scalar potential, the Helmholtz free energy $\hat{f}_c(\mathbf{c}^C, \mathbf{c}^R, \lambda)$, through a logarithmic coupling function

$$\Phi(\xi) = \frac{G_R}{2m\rho}\,a\,\xi^m \log(\xi^m), \qquad \xi = \lambda e^{-I_R}, \qquad I_R = \log|\mathbf{c}^R| - \operatorname{tr}(\mathbf{c}^R)$$

The constant $a$ is not a free fitting parameter: it is fixed analytically by imposing that the equilibrium (no-flow) state $\lambda=1$, $\mathbf{c}^R=\boldsymbol{\delta}$ ($I_R=-3$) is a critical point of $\hat{f}_c$, i.e. $\left.\partial\hat{f}_c/\partial\lambda\right|_{\lambda=1,\,I_R=-3}=0$.

## Workflow
In the following cells, we will: (1) symbolically construct the Mt-ETV free energy and solve for the equilibrium constant $a$ using SymPy, (2) derive the constitutive stress relations, relaxation functions, and entropy generation terms from $\hat{f}_c$ via symbolic differentiation, (3) lambdify the equations to obtain efficient numerical functions, (4) solve the resulting differential equations via time integration, and (5) generate comprehensive visualizations and store the results for multiple shear rates to characterize the model's predictions, for direct comparison against the original t-ETV model.

In [1]:
# Core symbolic and numerical libraries
import sympy as sp
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from extras import show

sp.init_printing()

# LaTeX-rendered plotting style, matching the t-ETV reference notebook
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "text.latex.preamble": r"\usepackage{amsfonts}\usepackage{amsmath}"
})

## 1. Symbolic Construction

We define the symbolic variables that will be used throughout the Mt-ETV model:

- **Independent variable**: time $t$
- **Imposed shear rate**: $\dot{\gamma}_0$ (constant for simple shear flow startup)
- **State variables**: components of the conformation tensors ($c^C_{ij}$, $c^R_{ij}$) and the structure parameter ($\lambda$) — identical set to the t-ETV model, since the underlying kinematic state is unchanged
- **Model parameters**: material constants governing viscosity, elasticity, and structural relaxation. Compared to t-ETV, the empirical kinetic parameters $\tau_{\rm aggr}$, $\tau_{\rm break}$, $d$ are absent — they are replaced by the coupling function $\Phi(\xi)$ and mobility $M(\lambda)$, both derived analytically from the Helmholtz free energy $\hat{f}_c$. The rouleaux dashpot viscosity $\mu_R$ is likewise replaced by a constant relaxation time $\tau_R$, consistent with the free-energy-derived relaxation function chosen for $\dot{\mathbf{c}}^R_{\rm relax}$.

In [2]:
# --- independent variable and imposed shear rate ---
t = sp.symbols('t', positive=True)
gamma0 = sp.symbols(r'\dot{\gamma_0}', real=True)   # dot(gamma)_0, constant (simple shear startup)

# --- state: components of conformation tensors + lambda ---
# Same state variables as the t-ETV model: the kinematic description is unchanged,
# only the constitutive relations governing their evolution differ.
cCxx, cCxy, cCyy, cCzz = sp.symbols(r'c^C_{xx} c^C_{xy} c^C_{yy} c^C_{zz}', real=True)
cRxx, cRxy, cRyy, cRzz = sp.symbols(r'c^R_{xx} c^R_{xy} c^R_{yy} c^R_{zz}', real=True)
lam = sp.symbols(r'\lambda', positive=True)

state_syms = (cCxx, cCxy, cCyy, cCzz, cRxx, cRxy, cRyy, cRzz, lam)

# --- model parameters ---
# Cellular viscosity and moduli: identical to t-ETV
G_C, muC0, muCinf, tauC = sp.symbols(r'G_C \mu_{0-c} \mu_{\infty-C} \tau_C', positive=True)
G_R = sp.symbols(r'G_R', positive=True)

# Rouleaux structural relaxation: tau_R replaces the t-ETV dashpot viscosity mu_R,
# consistent with the free-energy-derived relaxation function for c^R_relax
tau_R = sp.symbols(r'\tau_R', positive=True)

# Power-law exponent, structural relaxation timescale for lambda
m = sp.symbols('m', positive=True)
tau_lam = sp.symbols(r'\tau_\lambda', positive=True)

# Thermodynamic state parameters
rho, Temp = sp.symbols('rho T', positive=True)

# Note: tau_break, tau_aggr, and d are NOT defined here — in the Mt-ETV model
# their role is absorbed into the coupling function Phi(xi) and mobility M(lambda),
# derived symbolically in the next section. The equilibrium constant 'a' is likewise
# not an independent fitted parameter; it will be solved for as a function of 'm'
# from the free-energy equilibrium condition.
param_syms = (G_C, muC0, muCinf, tauC, G_R, tau_R, m, tau_lam, rho, Temp)

### Cellular Viscosity (Cross-like Model)

The cellular contribution to the viscosity is **unchanged from the t-ETV model**: the thermodynamic derivation only modifies the constitutive relations tied to $\hat{f}_c$ (the rouleaux stress, the relaxation functions, and the structural evolution of $\lambda$), while the cellular viscosity remains an independently specified shear-rate-dependent (shear-thinning) closure:

$$\eta_C(\dot{\gamma}_0) = \mu_{C,\infty} + \frac{\mu_{C,0} - \mu_{C,\infty}}{1 + \tau_C \dot{\gamma}_0}$$

where $\mu_{C,0}$ is the zero-shear viscosity, $\mu_{C,\infty}$ is the infinite-shear viscosity, and $\tau_C$ is the relaxation time.

In [3]:
# Viscosity non-linear (Cross-like) for individual cellular contribution
# Identical closure to the t-ETV model: this term is not derived from the
# free energy and carries over unchanged into the Mt-ETV formulation.
eta_C = muCinf + (muC0 - muCinf)/(1 + tauC*gamma0)

show(eta_C)

<IPython.core.display.Math object>

## Complete Evolution Equations (Kinematic + Relaxation)

The kinematics are identical to the t-ETV model: same simple shear flow, same velocity gradient $\nabla\vec{v}$, same block-diagonal structure for $\mathbf{c}^C$ and $\mathbf{c}^R$, and the same kinematic (upper-convected) contribution to each component.

The relaxation terms are where the models diverge:

- **Cellular tensor**: unchanged from t-ETV — $\dot{\mathbf{c}}^C_{\rm relax} = \dfrac{G_C}{\eta_C(\dot\gamma_0)}(\boldsymbol{\delta}-\mathbf{c}^C)$, since this closure does not depend on the free energy.
- **Rouleaux tensor**: instead of a constant $1/\tau_R$, we use the free-energy-consistent relaxation function
$$\dot{\mathbf{c}}^R_{\rm relax} = -\frac{2\rho}{G_R\tau_R}\,\frac{\partial\hat{f}_c}{\partial I_R}\,(\boldsymbol{\delta}-\mathbf{c}^R)$$
This requires $\partial\hat{f}_c/\partial I_R$, so — unlike in the t-ETV notebook — we must introduce the free-energy machinery ($I_R$, $\xi$, $\Phi(\xi)$, and the equilibrium constant $a$) at this point rather than later, since t-ETV never needs it. The constant $a$ is solved symbolically here from the critical-point condition $\left.\partial\hat{f}_c/\partial\lambda\right|_{\lambda=1,I_R=-3}=0$, so no hand-derived formula is hardcoded.

In [4]:
# ============================================================
# Simple shear flow kinematics (identical to t-ETV)
# ============================================================
nabla_v = sp.Matrix([
    [0, gamma0, 0],
    [0, 0, 0],
    [0, 0, 0]
])
strain_rate = nabla_v + nabla_v.T
delta = sp.eye(3)

# Conformation tensor components; block-diagonal structure preserved by simple shear
c11C, c12C, c22C, c33C = sp.symbols('c_{11}^C c_{12}^C c_{22}^C c_{33}^C', real=True)
c11R, c12R, c22R, c33R = sp.symbols('c_{11}^R c_{12}^R c_{22}^R c_{33}^R', real=True)

cC = sp.Matrix([[c11C, c12C, 0], [c12C, c22C, 0], [0, 0, c33C]])
cR = sp.Matrix([[c11R, c12R, 0], [c12R, c22R, 0], [0, 0, c33R]])

# Upper-convected kinematic contribution: (grad v)^T . c + c . grad v
upper_conv_C = nabla_v.T @ cC + cC @ nabla_v
upper_conv_R = nabla_v.T @ cR + cR @ nabla_v

# ============================================================
# Cellular relaxation: unchanged from t-ETV (no free-energy dependence)
# ============================================================
relaxC_coef = G_C / eta_C
cC_relax = relaxC_coef * (delta - cC)

dc11C = upper_conv_C[0, 0] + cC_relax[0, 0]
dc12C = upper_conv_C[0, 1] + cC_relax[0, 1]
dc22C = upper_conv_C[1, 1] + cC_relax[1, 1]
dc33C = upper_conv_C[2, 2] + cC_relax[2, 2]

# ============================================================
# Free-energy machinery, needed here for the rouleaux relaxation function.
# I_R_sym is kept abstract for differentiation, then substituted with the
# actual invariant expression afterward (avoids hand-derived chain rule errors).
# ============================================================
I_R_sym = sp.symbols('I_R', real=True)
xi_sym = lam * sp.exp(-I_R_sym)
a = sp.symbols('a', positive=True)

# Lambda/I_R-dependent part of f_c (the cellular term drops out of these derivatives)
Phi = (G_R / (2*rho)) * a * xi_sym**m * sp.log(1 + xi_sym)
f_c_lam_IR = -(G_R / (2*m*rho)) * lam**m + Phi

# Solve the equilibrium constant 'a' from the critical-point condition at
# lambda = 1, I_R = -3 (undeformed, fully aggregated rouleaux state)
dfc_dlam = sp.diff(f_c_lam_IR, lam)
a_solution = sp.simplify(sp.solve(sp.Eq(dfc_dlam.subs({lam: 1, I_R_sym: -3}), 0), a)[0])
show(a_solution)   # expected: exp(-3*m)/(3*m + 1)

f_c_lam_IR = f_c_lam_IR.subs(a, a_solution)

# Substitute the actual invariant expression and differentiate w.r.t. I_R
I_R_expr = sp.log(cR.det()) - sp.trace(cR)
dfc_dIR = sp.diff(f_c_lam_IR, I_R_sym).subs(I_R_sym, I_R_expr)

# ============================================================
# Rouleaux relaxation: free-energy-consistent form (untested option)
# ============================================================
relaxR_coef = -(2*rho / (G_R*tau_R)) * dfc_dIR
cR_relax = relaxR_coef * (delta - cR)

dc11R = sp.simplify(upper_conv_R[0, 0] + cR_relax[0, 0])
dc12R = sp.simplify(upper_conv_R[0, 1] + cR_relax[0, 1])
dc22R = sp.simplify(upper_conv_R[1, 1] + cR_relax[1, 1])
dc33R = sp.simplify(upper_conv_R[2, 2] + cR_relax[2, 2])

print("Cellular tensor evolution (unchanged from t-ETV):")
show(dc11C); show(dc12C); show(dc22C); show(dc33C)

print("\nRouleaux tensor evolution (free-energy-consistent relaxation):")
show(dc11R); show(dc12R); show(dc22R); show(dc33R)

<IPython.core.display.Math object>

Cellular tensor evolution (unchanged from t-ETV):


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...


Rouleaux tensor evolution (free-energy-consistent relaxation):


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Structural Evolution Equation

The structure parameter $\lambda$ is no longer governed by a phenomenological breakup/aggregation law. Instead, it follows directly from the Helmholtz free energy through the kinematic coupling tensor and an Onsager-type relaxation:

$$\frac{d\lambda}{dt} = \underbrace{-2\lambda\dot{\gamma}_0 c_{12}^R}_{\mathbf{g}:\dot{\boldsymbol\gamma},\ \ \mathbf{g}=-\lambda(\mathbf{c}^R-\boldsymbol\delta)} \;+\; \underbrace{-M(\lambda)\,\frac{\partial\hat{f}_c}{\partial\lambda}}_{\dot\lambda_{\rm relax}}, \qquad M(\lambda) = \frac{2\rho}{G_R\tau_\lambda}$$

Here $\partial\hat f_c/\partial\lambda$ is obtained by symbolic differentiation of the free energy defined earlier (with the equilibrium constant $a$ already substituted), so no hand-derived expression is hardcoded.

Note that, unlike t-ETV, $\tau_R$ is **not** dynamically coupled to $d\lambda/dt$ here — it was already introduced as a constant material parameter in the rouleaux relaxation function derived in the previous cell, so no separate $1/\tau_R(\lambda,\dot\gamma)$ closure is needed.

In [16]:
# Onsager mobility function -- Proposal 3's PAIRED degenerate mobility,
# not the constant M_lam used before. The lambda^(1-m) factor exactly
# cancels the lambda^(m-1) singularity in df_c/dlambda as lambda -> 0,
# and (1-lambda) makes the relaxation vanish at full aggregation too.
M_lam = (2*rho / (G_R*tau_lam)) * lam**(1 - m) * (1 - lam)

dfc_dlam = sp.diff(f_c_lam_IR, lam).subs(I_R_sym, I_R_expr)
dlam_relax = -M_lam * dfc_dlam
dlam_kin = -2*lam*gamma0*c12R
dlam_dt = sp.simplify(dlam_kin + dlam_relax)

show(dlam_dt)

<IPython.core.display.Math object>

### Stress Tensor Components

The stress tensor has exactly the **same functional form** as in the t-ETV model. As shown in the derivation, the rouleaux stress is recovered identically for *any* differentiable coupling function $\Phi$ — the extra terms coming from $\partial\hat f_c/\partial\mathbf{c}^R$ and $\partial\hat f_c/\partial\lambda\cdot\mathbf{g}$ cancel exactly, leaving:

$$\tau_{xy} = \tau^C_{xy} + \tau^R_{xy}$$

where:
$$\tau^C_{xy} = -\eta_C(\dot{\gamma}_0)\,\dot{\gamma}_0 - G_C\,c^C_{xy}$$
$$\tau^R_{xy} = -G_R\,\lambda^m\,c^R_{xy}$$

This is expected: the free energy $\hat f_c$ was constructed precisely so that the stress constitutive relation matches the original t-ETV by design, regardless of the choice of $\Phi(\xi)$. The logarithmic coupling only affects the *evolution* of $\lambda$ and $\mathbf{c}^R$ (already derived), not the stress closure itself.

In [17]:
# Stress components — same functional form as t-ETV (the Phi(xi) choice does not
# alter the stress relation, only the relaxation dynamics of lambda and c^R)
tau_xy_C = -eta_C*gamma0 - G_C*c12C
tau_xy_R = -G_R*lam**m*c12R
tau_xy   = tau_xy_C + tau_xy_R

show(tau_xy)

<IPython.core.display.Math object>

### Entropy Generation Rate (Mt-ETV, from the Helmholtz Free Energy)

Unlike the original t-ETV — which postulates $\hat{s}_c$ directly and computes $\sigma = \partial\hat{s}_c/\partial X : \dot{X}_{\rm relax}$ term by term — the Mt-ETV entropy production follows directly from the entropy balance derived earlier from $\hat{f}_c$:

$$S_{\rm gen} = -\frac{\rho}{T}\left[\frac{\partial\hat{f}_c}{\partial\mathbf{c}^C}:\dot{\mathbf{c}}^C_{\rm relax} + \frac{\partial\hat{f}_c}{\partial\mathbf{c}^R}:\dot{\mathbf{c}}^R_{\rm relax} + \frac{\partial\hat{f}_c}{\partial\lambda}\dot{\lambda}_{\rm relax}\right]$$

Each bracketed term is individually $\leq 0$ by the Second Law analysis (Section on relaxation inequalities), so the overall $-\rho/T$ prefactor guarantees $S_{\rm gen}\geq 0$. Every quantity needed here — $\dot{\mathbf{c}}^C_{\rm relax}$, $\dot{\mathbf{c}}^R_{\rm relax}$, $\partial\hat{f}_c/\partial I_R$, $\partial\hat{f}_c/\partial\lambda$, $\dot\lambda_{\rm relax}$ — was already built in previous cells; only $\partial\hat{f}_c/\partial\mathbf{c}^C$ and $\partial\hat{f}_c/\partial\mathbf{c}^R$ (via the chain rule through $I_R$) are new, and both follow directly from identities already established when matching the stress relation.

In [18]:
# ============================================================
# Entropy Generation Rate (Mt-ETV)
# ============================================================
# Fully reuses previously defined objects — no re-derivation of traces,
# inverses, or relaxation coefficients from scratch:
#   cC, cR, delta        -> conformation tensors and identity (tensor cell)
#   cC_relax, cR_relax   -> relaxation functions (tensor evolution cell)
#   dfc_dIR, dfc_dlam    -> free-energy derivatives (lambda evolution cell)
#   dlam_relax           -> structural relaxation term (lambda evolution cell)
# No trC/trR/trinvC/trinvR/logdet helper variables needed (t-ETV style):
# the matrix trace product directly encodes the same contraction.

# df_c/dc^C = (G_C/2rho)(delta - (c^C)^-1) -- same identity used earlier
# to match the cellular stress relation; reused here for entropy.
dfc_dcC = (G_C/(2*rho)) * (delta - cC.inv())
term1 = sp.simplify(-(rho/Temp) * sp.trace(dfc_dcC @ cC_relax))

# df_c/dc^R = dfc_dIR * ((c^R)^-1 - delta), via chain rule d(I_R)/d(c^R)
dfc_dcR = dfc_dIR * (cR.inv() - delta)
term2 = sp.simplify(-(rho/Temp) * sp.trace(dfc_dcR @ cR_relax))

# df_c/dlambda * lambdadot_relax, already available as dfc_dlam, dlam_relax
term3 = sp.simplify(-(rho/Temp) * (dfc_dlam * dlam_relax))

Sgen = term1 + term2 + term3
Sgen_terms = (term1, term2, term3)

show(term1)
show(term2)
show(term3)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 2. Lambdification: From Symbolic to Numerical

**Lambdification** converts SymPy symbolic expressions into fast, vectorized NumPy functions. This step is essential for efficient numerical integration.

We will lambdify:
1. The **right-hand side (RHS)** of the differential equations: the 9-component state vector evolution
2. The **shear stress** $\tau_{xy}$ for post-processing analysis
3. The **entropy generation rate** $S_{\text{gen}}$ and its individual contributions

**Input arguments** for the lambdified functions:
- **State variables** ($\mathbf{y}$): 9 components of $\mathbf{c}^C$, $\mathbf{c}^R$, and $\lambda$
- **Control parameter** ($\dot{\gamma}_0$): constant imposed shear rate
- **Material parameters** (10 values): $G_C$, $\mu_{C,0}$, $\mu_{C,\infty}$, $\tau_C$, $G_R$, $\tau_R$, $m$, $\tau_\lambda$, $\rho$, $T$

Compared to t-ETV, there are **no** $\mu_R$, $\tau_{\rm break}$, $\tau_{\rm aggr}$, or $d$ — those empirical kinetic parameters are absent by construction in the Mt-ETV model; their role is played entirely by the free-energy-derived expressions already embedded symbolically in `dc11R...dc33R` and `dlam_dt`. The equilibrium constant $a$ does **not** appear as an argument either, since it was already resolved analytically (as a function of $m$) and substituted before these expressions were built.

The numerical solver will evaluate these functions at each time step to advance the solution.

In [19]:
# Construct the right-hand side state vector
# (Reuses dc11C...dc33R from the tensor-evolution cell and dlam_dt from the
# structural-evolution cell -- nothing is redefined here.)
rhs_syms = sp.Matrix([dc11C, dc12C, dc22C, dc33C,
                       dc11R, dc12R, dc22R, dc33R,
                       dlam_dt])

# Define argument order: state variables + shear rate + parameters
state_names = ['c11C', 'c12C', 'c22C', 'c33C', 'c11R', 'c12R', 'c22R', 'c33R', 'lambda']
param_names = ['G_C', 'muC0', 'muCinf', 'tauC', 'G_R', 'tau_R', 'm', 'tau_lam', 'rho', 'T']

# Lambdify all functions
_args = (c11C, c12C, c22C, c33C, c11R, c12R, c22R, c33R, lam, gamma0) + \
        (G_C, muC0, muCinf, tauC, G_R, tau_R, m, tau_lam, rho, Temp)

rhs_func         = sp.lambdify(_args, rhs_syms, modules='numpy')
tauxy_func       = sp.lambdify(_args, tau_xy, modules='numpy')
sgen_func        = sp.lambdify(_args, Sgen, modules='numpy')
sgen_terms_func  = sp.lambdify(_args, Sgen_terms, modules='numpy')

# Bonus: standalone function for the equilibrium constant a(m), useful for
# reporting/validation (e.g. confirming a ≈ 0.00738 for m = 3/2, since
# a = exp(-3m)/(3m+1) here -- no extra m factor).
a_func = sp.lambdify(m, a_solution, modules='numpy')

print("Lambdification complete.")
print(f"State variables: {state_names}")
print(f"Parameters: {param_names}")

Lambdification complete.
State variables: ['c11C', 'c12C', 'c22C', 'c33C', 'c11R', 'c12R', 'c22R', 'c33R', 'lambda']
Parameters: ['G_C', 'muC0', 'muCinf', 'tauC', 'G_R', 'tau_R', 'm', 'tau_lam', 'rho', 'T']


In [20]:
# Material parameters — Mt-ETV (logarithmic coupling model)
# Cellular parameters, G_R, m, rho, T: unchanged from t-ETV donor-average fit.
# tau_R and tau_lam: initial guesses derived from/inherited from t-ETV, to be
# refined later via parameter optimization against the t-ETV reference results.
param_dict = {
    'G_C': 0.5471,               # Pa    - elastic modulus for individual cells
    'muC0': 0.0082,              # Pa·s  - zero-shear viscosity
    'muCinf': 0.0034,            # Pa·s  - infinite-shear viscosity
    'tauC': 0.0474,              # s     - Cross model characteristic time
    'G_R': 0.1552,               # Pa    - elastic modulus for rouleaux
    'tau_R': 0.0390 / 0.1552,    # s     - rouleaux relaxation time (initial guess = muR/G_R from t-ETV; expected to need refitting)
    'm': 1.5,                    # -     - power-law exponent
    'tau_lam': 1.9819,           # s     - structure relaxation time (initial guess from t-ETV; expected to need refitting)
    'rho': 1060.0,               # kg/m³ - blood density
    'T': 310.0,                  # K     - body temperature
}

## Flow Protocol Simulations

We will simulate the Mt-ETV model (logarithmic coupling, free-energy-consistent $\dot{\mathbf{c}}^R_{\rm relax}$) under the same three flow protocols used for t-ETV, for direct comparability:

1. **Startup flow**: Constant shear rate $\dot{\gamma}_0$ in the $+x$ direction applied from $t=0$
2. **Stress relaxation**: Shear flow stopped ($\dot{\gamma}_0 \to 0$) after reaching steady state
3. **Reversal flow**: Shear rate suddenly reversed from $+\dot{\gamma}_0$ to $-\dot{\gamma}_0$ at a fixed time

For each protocol, we track the same three quantities as t-ETV:
- **Structure parameter** ($\lambda$): aggregation state
- **Shear stress** ($\tau_{xy}$): viscous response
- **Entropy generation** ($\dot{S}_{\text{gen}}$): thermodynamic dissipation

Results are saved to a dedicated output directory (`MtETV_log`, distinct from the earlier non-working Mt-ETV attempt) for later comparison against the t-ETV reference and, eventually, against the $\Phi=0$ reduction.

In [27]:
# Convert to ordered tuple matching _args order (state + gamma0 + params)
params_tuple = tuple(param_dict[name] for name in param_names)

# Initial condition: identity tensors + full aggregation (same as t-ETV)
y0 = np.array([1.0, 0.0, 1.0, 1.0,      # c^C: (c11, c12, c22, c33)
               1.0, 0.0, 1.0, 1.0,       # c^R: (c11, c12, c22, c33)
               1.0])                     # lambda

print(f"Initial condition: {y0}")
print(f"Parameters tuple created: {len(params_tuple)} values\n")

# Create output directories
# Using a distinct name (MtETV_log) to keep this rebuild separate from any
# earlier, non-working Mt-ETV notebook output.
import os
OUTPUT_DIR = 'data/MtETV_log'
FIGURE_DIR = 'figs/MtETV_log'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

# Shear rates to simulate (same as t-ETV, for direct comparison)
gamma0_list = [2.0, 5.0, 10.0]  # 1/s

# Time integration settings (same solver/tolerances as t-ETV)
INTEGRATOR_OPTS = {'method': 'Radau', 'rtol': 1e-8, 'atol': 1e-10}

print("Setup complete.")

Initial condition: [1. 0. 1. 1. 1. 0. 1. 1. 1.]
Parameters tuple created: 10 values

Setup complete.


## Protocol 1: Startup Flow

We apply a constant shear rate $\dot{\gamma}_0$ starting from rest, using the Mt-ETV logarithmic-coupling model derived above.

**Time domain**: $t \in [0, 10]$ s with logarithmic resolution.

In [28]:
def rhs_wrapper(t, y_state, gamma0_val, params):
    """Wrapper to call rhs_func with correct argument order"""
    return np.array(rhs_func(*y_state, gamma0_val, *params), dtype=float).flatten()

# Time setup (identical to t-ETV, for direct comparability)
t_startup_span = (0.0, 10.0)
t_startup_eval = np.concatenate([[0.0], np.logspace(-3, np.log10(10.0), 1000)])

startup_results = {}

print("Running startup flow (Mt-ETV, logarithmic coupling)...")
for g0 in gamma0_list:
    sol = solve_ivp(
        lambda t, y: rhs_wrapper(t, y, g0, params_tuple),
        t_startup_span,
        y0,
        t_eval=t_startup_eval,
        **INTEGRATOR_OPTS
    )

    # Flag failed integrations early -- the free-energy-consistent c^R relaxation
    # involves log(det(c^R)) and (c^R)^-1, which are more prone to numerical
    # trouble than t-ETV's simple linear relaxation if c^R loses positive-definiteness.
    if not sol.success:
        print(f"  WARNING: integration failed for γ̇₀ = {g0} 1/s -- {sol.message}")

    # Vectorized post-processing: tauxy_func / sgen_func are scalar SymPy
    # expressions lambdified with modules='numpy', so they broadcast directly
    # over the full solution array -- no per-timestep Python loop needed.
    tauxy = np.asarray(tauxy_func(*sol.y, g0, *params_tuple), dtype=float)
    sgen = np.asarray(sgen_func(*sol.y, g0, *params_tuple), dtype=float)

    startup_results[g0] = {'t': sol.t, 'y': sol.y, 'tauxy': tauxy, 'sgen': sgen}
    print(f"  ✓ γ̇₀ = {g0} 1/s")

# Save data
startup_data = {'t': t_startup_eval}
for g0 in gamma0_list:
    r = startup_results[g0]
    startup_data[f'lam_g{g0}'] = r['y'][8, :]
    startup_data[f'tauxy_g{g0}'] = r['tauxy']
    startup_data[f'sgen_g{g0}'] = r['sgen']

np.savez(os.path.join(OUTPUT_DIR, 'startup.npz'), **startup_data)
print(f"Saved to {OUTPUT_DIR}/startup.npz\n")

Running startup flow (Mt-ETV, logarithmic coupling)...


KeyboardInterrupt: 

In [23]:
# ============================================================
# Instant check: is dlam_dt now bounded as lambda -> 0? (no integration)
# ============================================================
dlam_dt_relax_only = dlam_relax.subs({G_C: param_dict['G_C'], G_R: param_dict['G_R'],
                                        m: param_dict['m'], rho: param_dict['rho'],
                                        Temp: param_dict['T'], tau_lam: param_dict['tau_lam']})

print("dlam_relax as lambda -> 0, at I_R = -6.27 (the deformed state we saw fail before):")
for lam_val in [0.1, 0.01, 0.001, 0.0001, 1e-6]:
    val = dlam_dt_relax_only.subs({lam: lam_val, I_R_sym: -6.27, c11R: 1, c22R: 1, c12R: 1.8, c33R: 1})
    # I_R_sym was already substituted by I_R_expr earlier in dlam_relax, so this
    # subs on I_R_sym has no effect -- substitute the actual c^R components instead
    val2 = dlam_dt_relax_only.subs({lam: lam_val, c11R: 1, c22R: 4.5, c12R: 1.8, c33R: 1})
    print(f"  lambda={lam_val:<10} dlam_relax = {sp.N(val2):.6e}")

dlam_relax as lambda -> 0, at I_R = -6.27 (the deformed state we saw fail before):
  lambda=0.1        dlam_relax = -7.660617e+1
  lambda=0.01       dlam_relax = -4.330906e+1
  lambda=0.001      dlam_relax = -1.155804e+1
  lambda=0.0001     dlam_relax = -1.061683e+0
  lambda=1e-06      dlam_relax = 4.883342e-1


In [24]:
# ============================================================
# Cheap check (no integration): does df_c/dlambda have a SECOND root
# near lambda ~ 1e-5 at I_R = -6.27, or is the sign flip a float64
# precision artifact? Root-finding in u=log(lambda) space, as before,
# avoids the fractional-power cancellation problem entirely.
# ============================================================
import mpmath as mp
mp.mp.dps = 50  # 50 decimal digits -- eliminates float64 cancellation concerns

subs_params = {G_C: param_dict['G_C'], G_R: param_dict['G_R'], m: param_dict['m'],
               rho: param_dict['rho'], Temp: param_dict['T']}

dfc_dlam_IR_free = sp.diff(f_c_lam_IR, lam).subs(subs_params)
u_sym = sp.symbols('u', real=True)
dfc_du_IR_free = dfc_dlam_IR_free.subs(lam, sp.exp(u_sym))

I_R_diag = -6.27  # same state as the failing trajectory
dfc_du_at_IR = sp.lambdify(u_sym, dfc_du_IR_free.subs(I_R_sym, I_R_diag), modules='mpmath')

# Scan u = log(lambda) over a wide range and record sign changes
u_scan = [mp.mpf(v) for v in np.linspace(-25, 2, 400)]  # lambda from e^-25 (~1e-11) to e^2 (~7.4)
signs = []
for u_val in u_scan:
    try:
        val = dfc_du_at_IR(u_val)
        signs.append((float(u_val), float(np.exp(float(u_val))), float(val)))
    except Exception:
        signs.append((float(u_val), float(np.exp(float(u_val))), None))

print(f"{'u=log(lambda)':>15} | {'lambda':>12} | {'df_c/dlambda (high precision)':>30}")
print("-"*65)
prev_sign = None
for u_val, lam_val, val in signs:
    if val is None:
        continue
    curr_sign = val > 0
    marker = "  <-- SIGN CHANGE" if (prev_sign is not None and curr_sign != prev_sign) else ""
    if marker or lam_val < 1e-3 or abs(lam_val - 1.0) < 0.05:
        print(f"{u_val:15.3f} | {lam_val:12.4e} | {val:30.6e}{marker}")
    prev_sign = curr_sign

  u=log(lambda) |       lambda |  df_c/dlambda (high precision)
-----------------------------------------------------------------
        -25.000 |   1.3888e-11 |                  -2.728190e-10
        -24.932 |   1.4860e-11 |                  -2.822077e-10
        -24.865 |   1.5901e-11 |                  -2.919194e-10
        -24.797 |   1.7014e-11 |                  -3.019654e-10
        -24.729 |   1.8205e-11 |                  -3.123570e-10
        -24.662 |   1.9480e-11 |                  -3.231063e-10
        -24.594 |   2.0843e-11 |                  -3.342255e-10
        -24.526 |   2.2303e-11 |                  -3.457274e-10
        -24.459 |   2.3864e-11 |                  -3.576250e-10
        -24.391 |   2.5535e-11 |                  -3.699321e-10
        -24.323 |   2.7323e-11 |                  -3.826628e-10
        -24.256 |   2.9235e-11 |                  -3.958315e-10
        -24.188 |   3.1282e-11 |                  -4.094534e-10
        -24.120 |   3.3472e-11 |      

In [25]:
# ============================================================
# Reformulate in u = log(lambda) space, now with the Proposal-3 model
# (regularized Phi + paired degenerate mobility M(lambda)).
# Reuses dlam_dt, dc11R...dc33R, tau_xy, Sgen as currently defined.
# ============================================================

u_sym = sp.symbols('u', real=True)
lam_of_u = sp.exp(u_sym)

# Chain rule: du/dt = (dlambda/dt) / lambda
du_dt = sp.simplify(dlam_dt.subs(lam, lam_of_u) / lam_of_u)

dc11R_u = dc11R.subs(lam, lam_of_u)
dc12R_u = dc12R.subs(lam, lam_of_u)
dc22R_u = dc22R.subs(lam, lam_of_u)
dc33R_u = dc33R.subs(lam, lam_of_u)

tau_xy_u     = tau_xy.subs(lam, lam_of_u)
Sgen_u       = Sgen.subs(lam, lam_of_u)
Sgen_terms_u = tuple(term.subs(lam, lam_of_u) for term in Sgen_terms)

show(du_dt)

<IPython.core.display.Math object>

In [26]:
# ============================================================
# Re-lambdify with u replacing lambda
# ============================================================
_args = (c11C, c12C, c22C, c33C, c11R, c12R, c22R, c33R, u_sym, gamma0) + \
        (G_C, muC0, muCinf, tauC, G_R, tau_R, m, tau_lam, rho, Temp)

rhs_syms = sp.Matrix([dc11C, dc12C, dc22C, dc33C,
                       dc11R_u, dc12R_u, dc22R_u, dc33R_u,
                       du_dt])

rhs_func        = sp.lambdify(_args, rhs_syms, modules='numpy')
tauxy_func      = sp.lambdify(_args, tau_xy_u, modules='numpy')
sgen_func       = sp.lambdify(_args, Sgen_u, modules='numpy')
sgen_terms_func = sp.lambdify(_args, Sgen_terms_u, modules='numpy')

def rhs_wrapper(t, y_state, gamma0_val, params):
    return np.array(rhs_func(*y_state, gamma0_val, *params), dtype=float).flatten()

# Initial condition: lambda=1 -> u = log(1) = 0
y0 = np.array([1.0, 0.0, 1.0, 1.0,
               1.0, 0.0, 1.0, 1.0,
               0.0])

print("Re-lambdified in u=log(lambda) space (Proposal 3 model).")

Re-lambdified in u=log(lambda) space (Proposal 3 model).
